In [17]:
from pathlib import Path
import re
import matplotlib
import matplotlib.pyplot as plt
# %matplotlib notebook
%matplotlib inline
import numpy as np
import pandas as pd
import seaborn as sns  
import pickle as pkl
from tqdm import tqdm
from tensorboard.backend.event_processing import event_accumulator
import ot
import torch

from utils import load_hf_demos, get_trajectory_list

In [2]:
demos = load_hf_demos('Hopper-v3', folder = '../demos')

In [8]:
demo_list = get_trajectory_list(demos['all'])

In [11]:
pairs = [(a, b) for idx, a in enumerate(demo_list) for b in demo_list[idx + 1:]]

In [22]:
def compute_sw_barycenter(a,b,lr=1e3,nb_iter_max=1000,bary_size=1000,n_proj=100):
    x1_torch = torch.tensor(a)
    x2_torch = torch.tensor(b)
    xbinit = np.random.randn(bary_size, a.shape[-1])
    xbary_torch = torch.tensor(xbinit).requires_grad_(True)

    #x_all = np.zeros((nb_iter_max, xbary_torch.shape[0], 2))

    loss_iter = []

    # generator for random permutations
    gen = torch.Generator()
    gen.manual_seed(42)

    alpha = 0.5 # adaptive alpha schedule?

    for i in tqdm(range(nb_iter_max)):

        loss = alpha * ot.sliced_wasserstein_distance(xbary_torch, x2_torch, n_projections=n_proj, seed=gen) \
            + (1 - alpha) * ot.sliced_wasserstein_distance(xbary_torch, x1_torch, n_projections=n_proj, seed=gen)

        loss_iter.append(loss.clone().detach().cpu().numpy())
        loss.backward()

        # performs a step of projected gradient descent
        with torch.no_grad():
            grad = xbary_torch.grad
            xbary_torch -= grad * lr  # / (1 + i / 5e1)  # step
            xbary_torch.grad.zero_()
            #x_all[i, :, :] = xbary_torch.clone().detach().cpu().numpy()

    xb = xbary_torch.clone().detach().cpu().numpy()
    
    return xb
    

In [25]:
def enhance_demos_with_barycenters(demos,lr=1e3,nb_iter_max=1000,bary_size=1000,n_proj=100):
    demo_list = get_trajectory_list(demos)
    pairs = [(a, b) for idx, a in enumerate(demo_list) for b in demo_list[idx + 1:]]
    barys = []
    for a,b in pairs:
        barys.append(compute_sw_barycenter(a,b,lr,nb_iter_max,bary_size,n_proj))
    
    demo_list.extend(barys)
    
    return np.concatenate(demo_list,0)
    

In [23]:
barys = []
for a,b in pairs[:5]:
    barys.append(compute_sw_barycenter(a,b))


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:14<00:00, 69.84it/s]
